# **Atividade Prática**
<font size=3>

- **Tema:** amostragem e fluxo de trabalho.
---

### **1. Questão:**
<font size=3>

Com base no *dataset* de [Fraudes de cartões de crédito](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), disponível no diretório $\text{dataset/}\,$, realize os seguintes passos:
1. Importe os dados e **imprima na tela** a proporção de classes;
2. Faça um divisão estratificada de dados em treinamento, validação e teste;
3. Faça a busca do hiperparâmetro $k$, **em um laço** `for`, do modelo **k-NN**. Utilize os dados de validação para medir a performance do modelo a cada valor de $k$; notebook 6
4. **Imprima na tela** o melhor valor de $k$ e **retreine** o modelo com este valor. Utilize os dados de treinamento + validação para o *fit* do modelo;
5. Faça a avaliação final do melhor modelo com os dados de teste.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


In [ ]:
import pandas as pd
df = pd.read_csv ("/content/drive/MyDrive/IMD3002 - Aprendizado de Máquina Supervisionado/2-unidade/dataset/credit_card_fraud.csv", low_memory=True)
df = df.sample(frac=0.5)
df = df.drop(columns=['Unnamed: 0', 'index'])
print(df.head())

# separar X e y
X = df.drop('Class', axis=1).values
y = df['Class'].values

# proporção das classes
np.unique(y, return_counts=True)


          Time        V1        V2        V3        V4        V5        V6  \
3006  166450.0  1.938203 -0.096031 -0.990233  0.396266 -0.070444 -0.311743   
2815   91075.0 -1.855061  1.554964 -1.405809  0.669327 -0.280230  1.178652   
1898  109825.0  0.885544  0.717940 -2.388760  0.241808  3.313585  3.451589   
486    40985.0  1.335110 -0.455213  0.323804 -0.708680 -0.934714 -0.952303   
2166  127028.0  2.097362 -0.899636 -0.884913 -0.720271 -0.924084 -0.421369   

            V7        V8        V9  ...       V21       V22       V23  \
3006 -0.396549  0.060829  0.889654  ... -0.257102 -0.627846  0.293104   
2815 -3.459979 -2.815155  1.242229  ... -0.095308  0.946629 -0.297403   
1898  0.153211 -0.294185  0.512110  ...  0.828638  0.140478 -0.071546   
486  -0.394472 -0.198873 -1.273446  ...  0.295794  0.746610 -0.124953   
2166 -1.161425  0.054168  0.043811  ...  0.378245  1.179377  0.002354   

           V24       V25       V26       V27       V28  Amount  Class  
3006 -0.700195 -0.41

(array([0, 1]), array([1412,  256]))

In [ ]:
# dividindo os dados entre treino (60%) e de desenvolvimento (40%):
X_train, X_dev, y_train, y_dev = train_test_split(X, y, test_size=0.4, stratify=y)

# dividindo os dados de desenvolvimento entre validação (20%) e teste (20%):
X_val, X_test, y_val, y_test = train_test_split(X_dev, y_dev, test_size=0.5, stratify=y_dev)

print(f"X-train:{X_train.shape}, X-val:{X_val.shape}, X-test:{X_test.shape}")
print(f"y-train:{y_train.shape}, y-val:{y_val.shape}, y-test:{y_test.shape}")

X-train:(1000, 30), X-val:(334, 30), X-test:(334, 30)
y-train:(1000,), y-val:(334,), y-test:(334,)


In [ ]:
# verificando as proporções:
print(f"Proporção da classe 1 no treino: {y_train.mean():.2f}")
print(f"Proporção da classe 1 na validation: {y_val.mean():.2f}")
print(f"Proporção da classe 1 no teste: {y_test.mean():.2f}")

Proporção da classe 1 no treino: 0.15
Proporção da classe 1 na validation: 0.15
Proporção da classe 1 no teste: 0.16


In [ ]:
# normalização
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
# ajustar k usando o conjunto de validação:
scores = []

for k in range(1, 21):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)

    y_pred_val = model.predict(X_val)

    acc = accuracy_score(y_val, y_pred_val)

    scores.append(acc)

# escolher o melhor k:
best_k = np.argmax(scores)
print(f"Melhor k (validação): {best_k}")

Melhor k (validação): 3


In [ ]:
best_k_actual = best_k + 1
# modelo final
final_model = KNeighborsClassifier(n_neighbors=best_k_actual)
final_model.fit(np.concatenate([X_train, X_val]),
                np.concatenate([y_train, y_val]))

y_pred = final_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Acurácia no conjunto de teste: {acc:.2f}")

Acurácia no conjunto de teste: 0.97


### **2. Questão:**
<font size=3>

Com base no *dataset* de [Pinguins](https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data?select=penguins_size.csv), disponível no diretório $\text{dataset/}\,$, realize os seguintes passos:
1. Importe os dados e **imprima na tela** todas as classes das variáveis categóricas **e** suas proporções;
2. Caso exista alguma **classe indefinida**, veja quantas amostras esta classe apresenta. Caso seja um valor pequeno de amostras, remova a classe indefinida;
3. Observe se o *dataframe* apresenta valores `NaN`. Caso existam, remova a linha correspondente do *dataframe*;
4. Defina o atributo `species` como variável alvo (`y`), e as demais como `X`;
5. Realize as transformação necessárias nos dados categóricos e numéricos com base nas classes [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html) e [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html);
6. Utilize a abordagem correta de treinamento e avaliação do modelo de **Regressão Logística** com base no tamanho do *dataset* e a proporção de classes.
   

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv ("/content/drive/MyDrive/IMD3002 - Aprendizado de Máquina Supervisionado/2-unidade/dataset/penguins_size.csv", low_memory=True)
df = df.sample(frac=0.5, random_state=42)
df=  df.dropna()
df.head()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
194,Chinstrap,Dream,50.9,19.1,196.0,3550.0,MALE
157,Chinstrap,Dream,45.2,17.8,198.0,3950.0,FEMALE
225,Gentoo,Biscoe,46.5,13.5,210.0,4550.0,FEMALE
208,Chinstrap,Dream,45.2,16.6,191.0,3250.0,FEMALE
318,Gentoo,Biscoe,48.4,14.4,203.0,4625.0,FEMALE


In [ ]:
for col in df.select_dtypes(include='object').columns:
    print(f"\nColuna: {col}")
    print(df[col].value_counts(dropna=False))
    print(df[col].value_counts(normalize=True, dropna=False))


Coluna: species
species
Adelie       76
Gentoo       61
Chinstrap    30
Name: count, dtype: int64
species
Adelie       0.455090
Gentoo       0.365269
Chinstrap    0.179641
Name: proportion, dtype: float64

Coluna: island
island
Biscoe       82
Dream        57
Torgersen    28
Name: count, dtype: int64
island
Biscoe       0.491018
Dream        0.341317
Torgersen    0.167665
Name: proportion, dtype: float64

Coluna: sex
sex
MALE      87
FEMALE    79
.          1
Name: count, dtype: int64
sex
MALE      0.520958
FEMALE    0.473054
.         0.005988
Name: proportion, dtype: float64


In [ ]:
X = df.drop('species', axis=1)  # separação de x e y
y = df['species']
print (X.columns)

Index(['island', 'culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm',
       'body_mass_g', 'sex'],
      dtype='object')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.head()

,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
76,Torgersen,40.9,16.8,191.0,3700.0,FEMALE
66,Biscoe,35.5,16.2,195.0,3350.0,FEMALE
101,Biscoe,41.0,20.0,203.0,4725.0,MALE
72,Torgersen,39.6,17.2,196.0,3550.0,FEMALE
219,Dream,50.2,18.7,198.0,3775.0,FEMALE


In [ ]:
y_train.head()

,species
76,Adelie
66,Adelie
101,Adelie
72,Adelie
219,Chinstrap


In [ ]:
# definindo o codificador de etiquetas:
le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train) # aprende e transforma em números
y_test_encoded = le.transform(y_test) # transformação no teste

print("\nAlvo de treino depois da codificação:")
print(f"Classes aprendidas pelo LabelEncoder: {le.classes_}")
print("Alvos de treino depois da codificação:", y_train_encoded[:10])


Alvo de treino depois da codificação:
Classes aprendidas pelo LabelEncoder: ['Adelie' 'Chinstrap' 'Gentoo']
Alvos de treino depois da codificação: [0 0 0 0 1 0 0 2 0 2]


In [ ]:
# identificando as colunas numéricas e categóricas:
numerical_atributes =  X.select_dtypes(include=['int64', 'float64']).columns
categorical_atributes = X.select_dtypes(include=['object']).columns

# definindo os transformadores para cada tipo de dado:
numeric_transformer = StandardScaler() # normaliza os num
categorical_transformer = OneHotEncoder() # converte categorias em num - colunas binárias

# definindo o objeto ColumnTransformer para aplicar as transformações em X:
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numerical_atributes),
                                               ('cat', categorical_transformer, categorical_atributes)],
                                 remainder='passthrough' # categorial que não forem identificadas serão consideradas, mas sem transformação
                                )

# visualizando o objeto processador:
preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('num', StandardScaler(),
                                 Index(['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm',
       'body_mass_g'],
      dtype='object')),
                                ('cat', OneHotEncoder(),
                                 Index(['island', 'sex'], dtype='object'))])

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# definindo o pipeline:
pipe = Pipeline(steps=[('preprocessor', preprocessor),
                       ('classifier', LogisticRegression())])

pipe.fit(X_train, y_train_encoded) # treinando o pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  Index(['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm',
       'body_mass_g'],
      dtype='object')),
                                                 ('cat', OneHotEncoder(),
                                                  Index(['island', 'sex'], dtype='object'))])),
                ('classifier', LogisticRegression())])

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = pipe.predict(X_test)

acc = accuracy_score(y_test_encoded, y_pred)

print(f"Acurácia: {acc:.4f}")

Acurácia: 0.9706


### **3. Questão:**
<font size=3>

Com base no *dataset* de [Cogumelos](https://www.kaggle.com/datasets/uciml/mushroom-classification), disponível no diretório $\text{dataset/}\,$, realize os seguintes passos:
1. Importe os dados e **imprima na tela** a proporção da classificação definidas aos cogumelos (`class`);
2. Defina o atributo `class` como variável alvo (`y`), e as demais como `X`;
3. Faça a divisão dos *dataset* entre **treinamento** e **teste**;
4. Realize as transformação necessárias nos dados categóricos com base nas classes [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html) e [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html);
5. Defina um objeto `Pipeline` para encadear o pré-processamento de `X_train` junto ao treinamento do modelo **k-NN**;
6. Utilize a classe [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) para realizar a busca do melhor par de hiperparâmetros `n_neighbors` e `p` . Para esta busca, defina 20 iterações e 3 divisões para a validação cruzada;
7. Faça a previsão e avaliação do melhor modelo com os dados de teste .
   

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv ("/content/drive/MyDrive/IMD3002 - Aprendizado de Máquina Supervisionado/2-unidade/dataset/mushrooms.csv", low_memory=True)
df = df.sample(frac=0.5, random_state=42)
df=  df.dropna()
df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
1971,e,f,f,n,f,n,f,w,b,h,...,f,w,w,p,w,o,e,n,s,g
6654,p,f,s,e,f,y,f,c,n,b,...,s,p,p,p,w,o,e,w,v,l
5606,p,x,y,n,f,f,f,c,n,b,...,s,w,p,p,w,o,e,w,v,l
3332,e,f,y,g,t,n,f,c,b,n,...,s,g,p,p,w,o,p,n,y,d
6988,p,f,s,e,f,s,f,c,n,b,...,s,p,p,p,w,o,e,w,v,l


In [ ]:
print("Proporção das classes:")
print(df['class'].value_counts(normalize=True))

Proporção das classes:
class
e    0.513294
p    0.486706
Name: proportion, dtype: float64


In [ ]:
# definindo x e y
X = df.drop('class', axis=1)
y = df['class']

In [ ]:
# dividindo em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.head()

,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
4932,x,y,g,f,f,f,c,b,g,e,...,k,p,n,p,w,o,l,h,v,p
3123,x,y,e,t,n,f,c,b,n,t,...,s,w,g,p,w,o,p,k,v,d
4716,f,f,y,f,f,f,c,b,h,e,...,k,p,n,p,w,o,l,h,v,p
856,f,s,y,t,l,f,w,n,n,t,...,s,w,w,p,w,o,p,u,v,d
7705,x,s,p,t,n,f,c,b,w,e,...,s,w,w,p,w,t,p,w,v,p


In [ ]:
y_train.head()

,class
4932,p
3123,e
4716,p
856,e
7705,e


In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print("\nAlvo de treino depois da codificação:")
print(f"Classes aprendidas pelo LabelEncoder: {le.classes_}")
print("Alvos de treino depois da codificação:", y_train_enc[:10])


Alvo de treino depois da codificação:
Classes aprendidas pelo LabelEncoder: ['e' 'p']
Alvos de treino depois da codificação: [1 0 1 0 0 0 0 1 0 0]


In [ ]:
# identificando as colunas numéricas e categóricas:
categorical_atributes = X.select_dtypes(include=['object']).columns

categorical_transformer = OneHotEncoder(handle_unknown='ignore') # converte categorias em num - colunas binárias, ignorando categorias desconhecidas

# definindo o objeto ColumnTransformer para aplicar as transformações em X:
preprocessor = ColumnTransformer(transformers=[('cat', categorical_transformer, categorical_atributes)],
                                 remainder='passthrough' # categorial que não forem identificadas serão consideradas, mas sem transformação
                                )

# visualizando o objeto processador:
preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 Index(['cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor',
       'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color',
       'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
       'stalk-surface-below-ring', 'stalk-color-above-ring',
       'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number',
       'ring-type', 'spore-print-color', 'population', 'habitat'],
      dtype='object'))])

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

pipe = Pipeline(steps=[('preprocessor', preprocessor),
                       ('classifier', KNeighborsClassifier())])

param_grid = {
    'classifier__n_neighbors': np.arange(1, 21),
    'classifier__p': [1, 2]
}
# definindo o objeto RandomizedSearchCV:
random_search = RandomizedSearchCV(estimator=pipe,
                                   param_distributions=param_grid,
                                   n_iter=20,                      # testar 20 combinações aleatórias
                                   cv=3,                           # 3 divisões para validação cruzada
                                   scoring='accuracy',
                                   random_state=42,
                                   verbose=1)

# usando y_train_enc para o dataset de cogumelos
random_search.fit(X_train, y_train_enc)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('cat',
                                                                               OneHotEncoder(handle_unknown='ignore'),
                                                                               Index(['cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor',
       'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color',
       'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
       'stalk-s...
       'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number',
       'ring-type', 'spore-print-color', 'population', 'habitat'],
      dtype='object'))])),
                                             ('classifier',
                                              KNeighborsClassifier())]),
                   n_iter=20,
                   param_distributions={'classifier__n_neighbors': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20]),
                                        'classifier__p': [1, 2]},
                   random_state=42, scoring='accuracy', verbose=1)

In [ ]:
print("\nMelhores parâmetros:", random_search.best_params_)
print("Melhor score (validação):", random_search.best_score_)


Melhores parâmetros: {'classifier__p': 2, 'classifier__n_neighbors': np.int64(10)}
Melhor score (validação): 0.9996922129886118


In [ ]:
y_pred = random_search.predict(X_test)

acc = accuracy_score(y_test_enc, y_pred)
print(f"Acurácia no teste: {acc:.4f}")

Acurácia no teste: 0.9975
